# 01 · Quick exploration — PBDB Dinosauria
First look: volume, genera per period, distribution by continent. Data: `data/raw/dinos_pbdb.csv` (see README).

In [ ]:
import pandas as pd
df = pd.read_csv("../data/raw/dinos_pbdb.csv")
print(len(df), "occurrences |", df.genus.nunique(), "genera")

## Footprints and eggs
The PBDB includes ichnotaxa (footprints: *Grallator*, *Eubrontes*…) and ootaxa (eggs: *Megaloolithus*…). They are not biological species, so they are separated before counting diversity.

In [ ]:
trace_fam = ["Grallatoridae", "Eubrontidae", "Anomoepodidae", "Otozoidae", "Brontopodidae", "Iguanodontipodidae",
             "Elongatoolithidae", "Megaloolithidae", "Faveoloolithidae", "Dendroolithidae", "Spheroolithidae",
             "Prismatoolithidae", "Ovaloolithidae"]
df["is_trace"] = df.family.isin(trace_fam) | df.genus.str.contains(r"ool|pus$|podus$|ichnus$|^Grallator$|^Eubrontes$", regex=True, na=False)
df.is_trace.value_counts()

## Genera per period
Each fossil is assigned by the midpoint of its dating (`max_ma`/`min_ma`); the median uncertainty is ~8 Myr. Boundaries follow the PBDB time scale.

In [ ]:
bins   = [66, 100.5, 143.1, 161.5, 174.7, 201.4, 237, 247, 252]
labels = ["Late Cretaceous", "Early Cretaceous", "Late Jurassic", "Middle Jurassic", "Early Jurassic",
          "Late Triassic", "Middle Triassic", "Early Triassic"]
df["period"] = pd.cut((df.max_ma + df.min_ma) / 2, bins, labels=labels)
body = df[~df.is_trace]
body.groupby("period", observed=False).agg(occurrences=("genus", "size"), genera=("genus", "nunique")).iloc[::-1]

## Continent (approximate)
Rough lat/lng boxes, for orientation only. For real countries, re-download with `show=...,loc` (adds `cc`).

In [ ]:
def continent(r):
    la, lo = r.lat, r.lng
    if lo < -30: return "South America" if la < 12 else "North America"
    if la < -10 and lo > 110: return "Oceania"
    if -30 <= lo < 60: return "Africa" if la < 37 else "Europe"
    return "Asia"
df["continent"] = df.apply(continent, axis=1)
pd.crosstab(df.continent, df.period).iloc[:, ::-1]

In [ ]:
for p in ["Late Triassic", "Late Jurassic", "Late Cretaceous"]:
    print(p, body[body.period == p].family.replace("NO_FAMILY_SPECIFIED", None).value_counts().head(5).to_dict())